# MERIDA HRES — 1h Annual Maxima & Return Period Maps

**Pipeline overview**
1. Data inventory & integrity check
2. Single-file inspection (structure, units, CRS, time axis)
3. Extract 1h annual maxima (BM) across all files → `(years, y, x)` array
4. GEV fit pixel-wise with L-moments + MLE fallback
5. Compute RP quantile maps for [2, 5, 10, 25, 50, 100, 200] yr
6. Export GeoTIFFs (EPSG:4326 or native CRS)
7. Quick-look plots

**Assumptions / adjust to your setup**
- Files are `MERIDA_HRES_PREC_YYYYMM.nc`, one month each
- Variable name is `precipitation` (check §2 and change `PREC_VAR` if needed)
- Units are kg m⁻² h⁻¹ (= mm/h); if accumulated mm, set `IS_ACCUMULATED = True`
- All files on same grid; no reprojection needed inside the loop

## 0 · Imports & configuration

In [ ]:
import os
import re
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import rasterio
from rasterio.transform import from_bounds
from scipy.stats import genextreme as gev_scipy
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

# ── USER SETTINGS ──────────────────────────────────────────────────────────────
DATA_DIR       = Path("/home/admin_climatecharted_com/data/MERIDA_HRES")
OUT_DIR        = Path("./outputs/MERIDA_HRES_RP")
PREC_VAR       = "precipitation"   # change if needed (see §2)
IS_ACCUMULATED = False             # True → diff along time to get hourly increments
DURATION_H     = 1                 # 1-hour duration (sliding window or native timestep)
RETURN_PERIODS = [2, 5, 10, 25, 50, 100, 200]
N_JOBS         = 4                 # parallel workers for GEV fitting
# ───────────────────────────────────────────────────────────────────────────────

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUT_DIR.resolve()}")

## 1 · Data inventory

In [ ]:
files = sorted(DATA_DIR.glob("MERIDA_HRES_PREC_*.nc"))
print(f"Total files found: {len(files)}")

# Parse YYYYMM from filename
pat = re.compile(r"MERIDA_HRES_PREC_(\d{4})(\d{2})\.nc")
records = []
for f in files:
    m = pat.search(f.name)
    if m:
        records.append({"path": f, "year": int(m.group(1)), "month": int(m.group(2)),
                        "size_mb": f.stat().st_size / 1e6})

df_inv = pd.DataFrame(records).sort_values(["year","month"]).reset_index(drop=True)

print(f"\nYear range : {df_inv.year.min()} – {df_inv.year.max()}")
print(f"Months/year expected: 12")

# Check for missing months
years = range(df_inv.year.min(), df_inv.year.max()+1)
missing = []
for y in years:
    for mo in range(1,13):
        if not ((df_inv.year==y) & (df_inv.month==mo)).any():
            missing.append(f"{y}-{mo:02d}")

if missing:
    print(f"\n⚠ Missing months ({len(missing)}): {missing}")
else:
    print("\n✓ No missing months")

# File size distribution
print(f"\nFile size  min={df_inv.size_mb.min():.1f} MB  "
      f"max={df_inv.size_mb.max():.1f} MB  "
      f"mean={df_inv.size_mb.mean():.1f} MB")

# Flag suspiciously small files (possible corrupt/empty)
thresh = df_inv.size_mb.median() * 0.1
small  = df_inv[df_inv.size_mb < thresh]
if len(small):
    print(f"\n⚠ Suspiciously small files (< {thresh:.1f} MB):")
    print(small[["path","year","month","size_mb"]])
else:
    print("✓ No suspiciously small files")

df_inv.tail()

## 2 · Single-file deep inspection

In [ ]:
# Use the first file for inspection; repeat on a 'problem' file if needed
sample_path = df_inv.iloc[0].path
print(f"Inspecting: {sample_path.name}\n")

ds = xr.open_dataset(sample_path, engine="netcdf4")
print(ds)
print("\n--- Variables ---")
for v in ds.data_vars:
    d = ds[v]
    print(f"  {v:30s}  dims={d.dims}  shape={d.shape}  dtype={d.dtype}")
    if "units" in d.attrs: print(f"    units = {d.attrs['units']}")
    if "long_name" in d.attrs: print(f"    long_name = {d.attrs['long_name']}")

print("\n--- Coordinates ---")
for c in ds.coords:
    print(f"  {c:20s}  {ds[c].values[:3]} … {ds[c].values[-3:]}")

print("\n--- Global attributes ---")
for k,v in ds.attrs.items():
    print(f"  {k}: {v}")

In [ ]:
# Auto-detect precipitation variable if default doesn't exist
if PREC_VAR not in ds.data_vars:
    candidates = [v for v in ds.data_vars
                  if any(k in v.lower() for k in ["prec","precip","rain","tp","pr"])]
    if candidates:
        PREC_VAR = candidates[0]
        print(f"Auto-detected PREC_VAR = '{PREC_VAR}'")
    else:
        print(f"⚠ Could not detect precip variable. Available: {list(ds.data_vars)}")
        print("  → Set PREC_VAR manually in §0")
else:
    print(f"✓ PREC_VAR = '{PREC_VAR}' found")

# Time-axis inspection
t = ds.time.values
print(f"\nTime axis: {len(t)} steps")
print(f"  First : {pd.Timestamp(t[0])}")
print(f"  Last  : {pd.Timestamp(t[-1])}")
dt_hours = (pd.Timestamp(t[1]) - pd.Timestamp(t[0])).total_seconds() / 3600
print(f"  Δt    : {dt_hours:.2f} h")

if dt_hours != 1.0:
    print(f"\n⚠ Timestep is {dt_hours}h, not 1h — check DURATION_H or resample strategy")

# Spatial bounds
lat = ds["lat"].values if "lat" in ds.coords else ds["latitude"].values
lon = ds["lon"].values if "lon" in ds.coords else ds["longitude"].values
print(f"\nSpatial: lat [{lat.min():.3f} … {lat.max():.3f}]  "
      f"lon [{lon.min():.3f} … {lon.max():.3f}]")
print(f"Grid size: {len(lat)} × {len(lon)}")

ds.close()

In [ ]:
# Quick spatial and temporal plot of the sample file
ds = xr.open_dataset(sample_path)
da = ds[PREC_VAR]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Spatial mean
da.mean(dim=[d for d in da.dims if d != 'time']).plot(ax=axes[0])
axes[0].set_title("Hourly domain-mean precip (mm/h)")

# Spatial snapshot — time-mean
da.mean(dim="time").plot(ax=axes[1], cmap="Blues")
axes[1].set_title("Time-mean precip")

plt.tight_layout()
plt.savefig(OUT_DIR / "00_sample_inspection.png", dpi=120)
plt.show()
ds.close()

## 3 · Extract 1h annual maxima (Block Maxima)

In [ ]:
# ── Coordinate name normalisation helper ────────────────────────────────────
def get_latlon_names(ds):
    lat_names = ["lat","latitude","y"]
    lon_names = ["lon","longitude","x"]
    latname = next((n for n in lat_names if n in ds.coords), None)
    lonname = next((n for n in lon_names if n in ds.coords), None)
    return latname, lonname

# ── Determine grid shape from first file ─────────────────────────────────────
ds0 = xr.open_dataset(df_inv.iloc[0].path)
latname, lonname = get_latlon_names(ds0)
lat_ref = ds0[latname].values
lon_ref = ds0[lonname].values
ny, nx  = len(lat_ref), len(lon_ref)
print(f"Grid: {ny} × {nx}  (lat × lon)")
ds0.close()

# ── Loop over years ──────────────────────────────────────────────────────────
unique_years = sorted(df_inv.year.unique())
n_years = len(unique_years)
print(f"Processing {n_years} years: {unique_years[0]} – {unique_years[-1]}")

ann_max = np.full((n_years, ny, nx), np.nan, dtype=np.float32)

for yi, year in enumerate(unique_years):
    month_files = df_inv[df_inv.year == year].path.tolist()
    if len(month_files) < 12:
        print(f"  ⚠ {year}: only {len(month_files)} months — included but may bias low")

    # Concatenate all months of the year along time
    ds_yr = xr.open_mfdataset(month_files, combine="by_coords",
                               engine="netcdf4", parallel=False)
    da_yr = ds_yr[PREC_VAR]

    # If accumulated (e.g. total since model init), diff to get increments
    if IS_ACCUMULATED:
        da_yr = da_yr.diff(dim="time").clip(min=0)

    # Rolling sum for sub-hourly data (no-op if dt == 1h)
    # If DURATION_H > 1 and timestep is 1h, roll sum of DURATION_H steps
    if DURATION_H > 1:
        da_yr = da_yr.rolling(time=DURATION_H, min_periods=DURATION_H).sum()

    # Annual maximum
    ann_max[yi] = da_yr.max(dim="time").values

    ds_yr.close()
    if (yi+1) % 5 == 0 or yi == n_years-1:
        print(f"  {year} done ({yi+1}/{n_years})")

print("\n✓ Annual maxima extraction complete")
print(f"  ann_max shape: {ann_max.shape}   NaN count: {np.isnan(ann_max).sum()}")

In [ ]:
# Save ann_max as NetCDF for reuse (avoids rerunning the loop)
ds_am = xr.Dataset(
    {"ann_max_1h": (["year", latname, lonname], ann_max,
                    {"units": "mm", "long_name": "1-hour annual maximum precipitation"})},
    coords={"year": unique_years, latname: lat_ref, lonname: lon_ref}
)
am_path = OUT_DIR / "MERIDA_HRES_annmax_1h.nc"
ds_am.to_netcdf(am_path)
print(f"Saved: {am_path}")

### 3.1 · Annual maxima diagnostics

In [ ]:
# Domain-mean annual maxima trend
am_mean = np.nanmean(ann_max, axis=(1,2))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(unique_years, am_mean, 'o-', color='steelblue')
axes[0].set_title("Domain-mean annual max 1h precip")
axes[0].set_ylabel("mm/h")
axes[0].set_xlabel("Year")

# Spatial map of long-term mean of annual maxima
mean_map = np.nanmean(ann_max, axis=0)
im = axes[1].pcolormesh(lon_ref, lat_ref, mean_map, cmap="Blues", shading="auto")
plt.colorbar(im, ax=axes[1], label="mm/h")
axes[1].set_title("Mean annual max 1h precip")
axes[1].set_xlabel("Lon"); axes[1].set_ylabel("Lat")

# Spatial map of inter-annual std
std_map = np.nanstd(ann_max, axis=0)
im2 = axes[2].pcolormesh(lon_ref, lat_ref, std_map, cmap="Oranges", shading="auto")
plt.colorbar(im2, ax=axes[2], label="mm/h")
axes[2].set_title("Std of annual max 1h precip")
axes[2].set_xlabel("Lon")

plt.tight_layout()
plt.savefig(OUT_DIR / "01_annmax_diagnostics.png", dpi=120)
plt.show()

In [ ]:
# NaN / valid pixel census per year
valid_pct = np.mean(~np.isnan(ann_max), axis=(1,2)) * 100
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(unique_years, valid_pct, color="seagreen")
ax.axhline(99, color="red", linestyle="--", label="99% threshold")
ax.set_ylabel("% valid pixels")
ax.set_xlabel("Year")
ax.set_title("Data completeness per year (annual maxima)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "02_data_completeness.png", dpi=120)
plt.show()

if (valid_pct < 95).any():
    bad_years = [y for y, v in zip(unique_years, valid_pct) if v < 95]
    print(f"⚠ Years with < 95% valid pixels: {bad_years}")

## 4 · GEV fitting (pixel-wise)

In [ ]:
# ── L-moments estimator for GEV (fast, used as initial guess / primary) ──────
def gev_lmom(data):
    """L-moments GEV fit. Returns (loc, scale, shape) in scipy convention (shape = -k)."""
    x = np.sort(data[~np.isnan(data)])
    n = len(x)
    if n < 10:
        return np.nan, np.nan, np.nan
    # Probability-weighted moments
    i = np.arange(1, n+1)
    b0 = np.mean(x)
    b1 = np.sum((i-1) / (n*(n-1)) * x)
    b2 = np.sum((i-1)*(i-2) / (n*(n-1)*(n-2)) * x)
    l1 = b0
    l2 = 2*b1 - b0
    l3 = 6*b2 - 6*b1 + b0
    t3 = l3 / l2  # L-skewness
    # Hosking (1985) approximation
    c = 2/(3+t3) - np.log(2)/np.log(3)
    k = 7.8590*c + 2.9554*c**2          # GEV shape (k > 0 → heavy tail)
    k = np.clip(k, -0.5, 0.5)           # stability guard (reuse from MORE pipeline)
    g1 = np.math.gamma(1+k) if abs(k) < 1 else 1.0
    alpha = l2 * k / ((1 - 2**(-k)) * g1) if abs(k) > 1e-6 else l2 / np.log(2)
    xi = l1 - alpha/k * (1 - g1)        # location
    # scipy convention: shape = -k, so Frechet/heavy tail → positive shape
    shape_scipy = -k
    return xi, alpha, shape_scipy


# ── MLE fallback ─────────────────────────────────────────────────────────────
def gev_mle(data):
    x = data[~np.isnan(data)]
    if len(x) < 10:
        return np.nan, np.nan, np.nan
    try:
        shape, loc, scale = gev_scipy.fit(x, method="MLE")
        shape = np.clip(shape, -0.5, 0.5)
        return loc, scale, shape
    except Exception:
        return np.nan, np.nan, np.nan


print("GEV fitting functions defined.")

In [ ]:
import math

gev_loc   = np.full((ny, nx), np.nan, dtype=np.float32)
gev_scale = np.full((ny, nx), np.nan, dtype=np.float32)
gev_shape = np.full((ny, nx), np.nan, dtype=np.float32)
fit_flag  = np.zeros((ny, nx), dtype=np.int8)   # 0=nan, 1=lmom, 2=mle, 3=failed

total_pixels = ny * nx
print(f"Fitting GEV for {total_pixels:,} pixels ({n_years} years each)…")

for j in range(ny):
    for i in range(nx):
        series = ann_max[:, j, i]
        valid  = series[~np.isnan(series)]
        if len(valid) < 10:
            fit_flag[j, i] = 0
            continue
        # Try L-moments first
        loc, sc, sh = gev_lmom(series)
        if np.isnan(loc) or sc <= 0:
            loc, sc, sh = gev_mle(series)
            fit_flag[j, i] = 2 if not np.isnan(loc) else 3
        else:
            fit_flag[j, i] = 1
        gev_loc[j, i]   = loc
        gev_scale[j, i] = sc
        gev_shape[j, i] = sh

n_ok     = np.sum(fit_flag > 0)
n_lmom   = np.sum(fit_flag == 1)
n_mle    = np.sum(fit_flag == 2)
n_fail   = np.sum(fit_flag == 3)
print(f"\n✓ Fit summary: {n_ok:,} pixels fitted")
print(f"  L-moments: {n_lmom:,}  |  MLE fallback: {n_mle:,}  |  Failed: {n_fail:,}")

In [ ]:
# GEV parameter maps
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
titles  = ["Location μ (mm/h)", "Scale σ (mm/h)", "Shape ξ"]
cmaps   = ["Blues", "Purples", "RdBu_r"]
params  = [gev_loc, gev_scale, gev_shape]

for ax, p, t, c in zip(axes, params, titles, cmaps):
    im = ax.pcolormesh(lon_ref, lat_ref, p, cmap=c, shading="auto")
    plt.colorbar(im, ax=ax, label=t)
    ax.set_title(t)
    ax.set_xlabel("Lon"); ax.set_ylabel("Lat")

plt.tight_layout()
plt.savefig(OUT_DIR / "03_gev_parameters.png", dpi=120)
plt.show()

# Shape parameter histogram
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(gev_shape[~np.isnan(gev_shape)].ravel(), bins=80, color="steelblue", edgecolor="none")
ax.axvline(0, color='k', linestyle='--', label='ξ=0 (Gumbel)')
ax.set_xlabel("Shape ξ"); ax.set_ylabel("Pixel count")
ax.set_title("GEV shape parameter distribution")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "04_gev_shape_histogram.png", dpi=120)
plt.show()

## 5 · Return period quantile maps

In [ ]:
def gev_quantile(loc, scale, shape, T):
    """GEV quantile for return period T years (exceedance prob p = 1/T)."""
    p = 1.0 - 1.0 / T
    y = -np.log(-np.log(p))            # reduced variate (Gumbel)
    with np.errstate(divide='ignore', invalid='ignore'):
        q = np.where(
            np.abs(shape) < 1e-6,
            loc + scale * y,           # Gumbel limit
            loc + scale / shape * (1 - np.exp(-shape * y))  # full GEV
        )
    q = np.where(np.isnan(loc), np.nan, q)
    return q.astype(np.float32)


rp_maps = {}
for T in RETURN_PERIODS:
    rp_maps[T] = gev_quantile(gev_loc, gev_scale, gev_shape, T)
    q_dom = np.nanmedian(rp_maps[T])
    q_p99 = np.nanpercentile(rp_maps[T], 99)
    print(f"  RP-{T:>4d}yr:  median={q_dom:.1f}  p99={q_p99:.1f}  mm/h")

In [ ]:
# Quick-look panel of all RP maps
ncols = 4
nrows = math.ceil(len(RETURN_PERIODS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
axes = axes.ravel()

# Common colour scale: 0 … 99th percentile of RP-100
vmax = np.nanpercentile(rp_maps[100], 99)

for ax, T in zip(axes, RETURN_PERIODS):
    im = ax.pcolormesh(lon_ref, lat_ref, rp_maps[T],
                       cmap="YlOrRd", vmin=0, vmax=vmax, shading="auto")
    plt.colorbar(im, ax=ax, label="mm/h")
    ax.set_title(f"RP {T} yr — 1h precip")
    ax.set_xlabel("Lon"); ax.set_ylabel("Lat")

for ax in axes[len(RETURN_PERIODS):]:
    ax.set_visible(False)

plt.suptitle("MERIDA HRES — 1h Return Period Precipitation (mm/h)", y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "05_RP_maps_panel.png", dpi=130, bbox_inches="tight")
plt.show()

## 6 · Export GeoTIFFs

In [ ]:
# Determine pixel resolution from grid
dlat = float(np.abs(np.diff(lat_ref).mean()))
dlon = float(np.abs(np.diff(lon_ref).mean()))

# rasterio needs (west, south, east, north)
west  = float(lon_ref.min()) - dlon/2
east  = float(lon_ref.max()) + dlon/2
south = float(lat_ref.min()) - dlat/2
north = float(lat_ref.max()) + dlat/2

transform = from_bounds(west, south, east, north, nx, ny)
crs_epsg  = "EPSG:4326"

# lat_ref may be descending; rasterio expects north-up
if lat_ref[0] < lat_ref[-1]:   # ascending → flip arrays
    needs_flip = True
    transform  = from_bounds(west, south, east, north, nx, ny)
else:
    needs_flip = False

profile = dict(
    driver="GTiff",
    dtype="float32",
    count=1,
    crs=crs_epsg,
    transform=transform,
    width=nx,
    height=ny,
    compress="lzw",
    nodata=np.nan,
    tiled=True,
    blockxsize=256,
    blockysize=256,
)

exported = []
for T, arr in rp_maps.items():
    data = np.flipud(arr) if needs_flip else arr
    out_path = OUT_DIR / f"MERIDA_HRES_1h_RP{T:03d}yr.tif"
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(data[np.newaxis, :, :])   # band 1
        dst.update_tags(return_period=f"{T} years",
                        duration="1 hour",
                        dataset="MERIDA HRES",
                        units="mm/h",
                        gev_method="L-moments + MLE fallback")
    exported.append(out_path)
    print(f"  Wrote: {out_path.name}  ({out_path.stat().st_size/1e6:.1f} MB)")

# Also export GEV parameters
for param_name, param_arr in [("loc", gev_loc), ("scale", gev_scale), ("shape", gev_shape)]:
    data = np.flipud(param_arr) if needs_flip else param_arr
    out_path = OUT_DIR / f"MERIDA_HRES_GEV_{param_name}.tif"
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(data[np.newaxis, :, :])
    exported.append(out_path)
    print(f"  Wrote: {out_path.name}")

print(f"\n✓ All GeoTIFFs saved to {OUT_DIR.resolve()}")

## 7 · Validation — empirical vs fitted at example point

In [ ]:
# Pick a point (Milan area if domain covers it; otherwise domain centroid)
POINT_LAT = 45.46   # change as needed
POINT_LON = 9.19

ji = np.argmin(np.abs(lat_ref - POINT_LAT))
ii = np.argmin(np.abs(lon_ref - POINT_LON))
actual_lat = lat_ref[ji]
actual_lon = lon_ref[ii]
print(f"Requested ({POINT_LAT:.2f}°N, {POINT_LON:.2f}°E) → "
      f"nearest grid ({actual_lat:.3f}°N, {actual_lon:.3f}°E)")

series_pt = ann_max[:, ji, ii]
loc_pt    = gev_loc[ji, ii]
sc_pt     = gev_scale[ji, ii]
sh_pt     = gev_shape[ji, ii]

# Weibull plotting positions
sorted_obs = np.sort(series_pt[~np.isnan(series_pt)])
n_obs = len(sorted_obs)
T_emp = (n_obs + 1) / np.arange(n_obs, 0, -1)  # empirical RP

# Fitted quantile curve
T_fit = np.logspace(np.log10(1.1), np.log10(500), 200)
Q_fit = gev_quantile(loc_pt, sc_pt, sh_pt, T_fit)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(T_emp, sorted_obs, color='steelblue', zorder=5, label='Observed annual maxima')
ax.plot(T_fit, Q_fit, 'r-', linewidth=2, label=f'GEV fit  ξ={sh_pt:.3f}')

# Mark target RPs
for T in RETURN_PERIODS:
    q = gev_quantile(loc_pt, sc_pt, sh_pt, T)
    ax.axvline(T, color='grey', linestyle=':', linewidth=0.8)
    ax.annotate(f"{q:.0f}", xy=(T, q), xytext=(2, 4), textcoords='offset points', fontsize=8)

ax.set_xscale('log')
ax.set_xlabel('Return period (years)')
ax.set_ylabel('1h precipitation (mm/h)')
ax.set_title(f'GEV validation — ({actual_lat:.3f}°N, {actual_lon:.3f}°E)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "06_gev_validation_point.png", dpi=120)
plt.show()

print(f"\nFitted GEV parameters at this point:")
print(f"  μ (location) = {loc_pt:.3f} mm/h")
print(f"  σ (scale)    = {sc_pt:.3f} mm/h")
print(f"  ξ (shape)    = {sh_pt:.4f}")
print("\nReturn period table (mm/h):")
for T in RETURN_PERIODS:
    print(f"  RP{T:>4d}yr = {gev_quantile(loc_pt, sc_pt, sh_pt, T):.1f} mm/h")

## 8 · Summary

In [ ]:
print("=" * 60)
print("MERIDA HRES — 1h Return Period Maps: Run Complete")
print("=" * 60)
print(f"\nDataset  : {df_inv.year.min()}–{df_inv.year.max()} ({n_years} years, {len(files)} monthly files)")
print(f"Grid     : {ny} × {nx} = {ny*nx:,} pixels")
print(f"Duration : {DURATION_H} h")
print(f"GEV fit  : L-moments primary, MLE fallback")
print(f"Pixels fitted : {n_ok:,} / {ny*nx:,}  ({100*n_ok/ny/nx:.1f}%)")
print(f"\nOutputs in {OUT_DIR.resolve()}:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:50s}  {f.stat().st_size/1e6:.1f} MB")